# 04 高维特征模型训练
本笔记本使用第二周构造的高维特征集 **test_augmented_D.csv** 和 **train_augmented_D.csv**，训练 3-4 个不同的生存分析模型：
随机生存森林 (RSF)、Gradient Boosting Survival (XGBoost)、多层感知机 / 深度生存模型 (DeepSurv)。

重点：
1. 不调参，先跑通基础结果。
2. 针对比赛 48 小时权重最高的特点，在模型训练中做优化，例如在损失中加权或者在提取验证时看48h结果。
3. 计算C指数、加权布里尔评分、混合分数，并整理成模型对比表。

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 我们假设你安装了 scikit-survival，如果用 DeepSurv 可使用 pycox
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, brier_score

train_df = pd.read_csv('data/train_augmented_D.csv').fillna(0)
test_df = pd.read_csv('data/test_augmented_D.csv').fillna(0)

# 构建标签，sksurv 要求的格式为 (event_indicator, event_time)
if 'event_id' in train_df.columns:
    train_df = train_df.drop(columns=['event_id'], errors='ignore')
if 'event' not in train_df.columns:
    train_df['event'] = True

# 修正：根据 csv 文件内容，生存时间列为 time_to_hit_hours
y_train = np.array([(bool(row['event']), row['time_to_hit_hours']) for _, row in train_df.iterrows()], dtype=[('event', '?'), ('time_to_hit_hours', '<f8')])
X_train = train_df.drop(columns=['time_to_hit_hours', 'event'], errors='ignore')

# 优化目标：48小时的预测精度 (可以构建 sample_weights 提供给支持的模型)
sample_weights = np.where(abs(train_df['time_to_hit_hours'] - 48) <= 24, 2.0, 1.0)

## 1. 评分函数及模型对比表 (从 00_metric 迁移)

In [19]:
results_table = []

def evaluate_model(model_name, model, X, y, y_train_dist):
    # 1. 计算 C-Index
    preds = model.predict(X_train)
    c_index = concordance_index_censored(y['event'], y['time_to_hit_hours'], preds)[0]
    
    # 2. 计算 Weighted Brier Score
    try:
        # scikit-survival的 Brier 评分计算不能超出训练集中最大的事件时间 (约66.9)
        # 否则会报 x must be within 错误。我们用 66 临时替代超出范围的 72
        max_valid_time = y['time_to_hit_hours'].max() - 0.1
        times = [24.0, 48.0, min(72.0, max_valid_time)]
        
        # 获取模型的生存概率预测函数
        surv_funcs = model.predict_survival_function(X)
        surv_probs = np.row_stack([fn(times) for fn in surv_funcs])
        
        _, brier_scores = brier_score(y_train_dist, y, surv_probs, times)
        
        # 权重: 24: 0.3, 48: 0.4, 72(或最大时间): 0.3
        weighted_brier = 0.3 * brier_scores[0] + 0.4 * brier_scores[1] + 0.3 * brier_scores[2]
        print(f"[{model_name}] Brier 分数计算成功！值为: {weighted_brier:.4f}")
    except Exception as e:
        print(f"Warning: {model_name} unable to compute Brier, using 0.5 (err: {e})")
        weighted_brier = 0.5
        
    # 3. 计算 混合分数
    mixed_score = 0.3 * c_index + 0.7 * (1 - weighted_brier)
    
    results_table.append({
        '模型名称': model_name,
        'C 指数': c_index,
        '加权布里尔评分': weighted_brier,
        '混合分数': mixed_score
    })
    print(f"[{model_name}] 评估完毕 -> 混合分数: {mixed_score:.4f}")

## 2. 训练各类模型并评测

### 模型预测：Random Survival Forest

In [20]:
print("Training Random Survival Forest...")
rsf = RandomSurvivalForest(n_estimators=50, min_samples_split=10, min_samples_leaf=15, n_jobs=-1, random_state=42)
# 若不支持 sample_weights，则常规 fit
rsf.fit(X_train, y_train)
evaluate_model('Random Survival Forest', rsf, X_train, y_train, y_train)

Training Random Survival Forest...
[Random Survival Forest] Brier 分数计算成功！值为: 0.0805
[Random Survival Forest] 评估完毕 -> 混合分数: 0.9218


### 模型预测：Gradient Boosting Survival Analysis

In [21]:
print("Training Gradient Boosting Survival Analysis...")
gbsa = GradientBoostingSurvivalAnalysis(n_estimators=50, learning_rate=0.1, random_state=42)
gbsa.fit(X_train, y_train)
evaluate_model('Gradient Boosting Survival / XGBoost', gbsa, X_train, y_train, y_train)

Training Gradient Boosting Survival Analysis...
[Gradient Boosting Survival / XGBoost] Brier 分数计算成功！值为: 0.0462
[Gradient Boosting Survival / XGBoost] 评估完毕 -> 混合分数: 0.9583


### 模型预测：DeepSurv / MLP Survival

In [22]:
print("Training DeepSurv / MLP Survival...")
# 神经网络对数据尺度最为敏感！务必先进行标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 这里我们用 Coxnet 模拟近似带有深度网络正则的 Cox，或如果你有 pycox 可以用:
# from pycox.models import DeepSurv， 此处写下占位和可运行逻辑
deepsurv_proxy = CoxnetSurvivalAnalysis(l1_ratio=0.1, alpha_min_ratio=0.01, fit_baseline_model=True)
deepsurv_proxy.fit(X_train_scaled, y_train)
evaluate_model('DeepSurv / MLP (Proxy)', deepsurv_proxy, X_train_scaled, y_train, y_train)

Training DeepSurv / MLP Survival...
[DeepSurv / MLP (Proxy)] Brier 分数计算成功！值为: 0.0882
[DeepSurv / MLP (Proxy)] 评估完毕 -> 混合分数: 0.9093


### 模型预测：Extra Survival Trees

In [23]:
from sksurv.ensemble import ExtraSurvivalTrees
print("Training Extra Survival Trees...")
est = ExtraSurvivalTrees(n_estimators=50, min_samples_split=10, min_samples_leaf=15, n_jobs=-1, random_state=42)
est.fit(X_train, y_train)
evaluate_model('Extra Survival Trees', est, X_train, y_train, y_train)

Training Extra Survival Trees...
[Extra Survival Trees] Brier 分数计算成功！值为: 0.1581
[Extra Survival Trees] 评估完毕 -> 混合分数: 0.8490


### 模型预测：Cox Proportional Hazards

In [24]:
from sksurv.linear_model import CoxPHSurvivalAnalysis
print("Training Cox Proportional Hazards...")
# Cox模型建议使用标准化后的数据防止不敛
cox = CoxPHSurvivalAnalysis(alpha=0.01)
cox.fit(X_train_scaled, y_train)
evaluate_model('Cox Proportional Hazards', cox, X_train_scaled, y_train, y_train)

Training Cox Proportional Hazards...
[Cox Proportional Hazards] Brier 分数计算成功！值为: 0.3120
[Cox Proportional Hazards] 评估完毕 -> 混合分数: 0.7529


### 模型预测：Component-wise Gradient Boosting

In [25]:
from sksurv.ensemble import ComponentwiseGradientBoostingSurvivalAnalysis
print("Training Component-wise Gradient Boosting...")
cwgb = ComponentwiseGradientBoostingSurvivalAnalysis(n_estimators=50, learning_rate=0.1, random_state=42)
cwgb.fit(X_train, y_train)
evaluate_model('Component-wise GBSA', cwgb, X_train, y_train, y_train)

Training Component-wise Gradient Boosting...
[Component-wise GBSA] Brier 分数计算成功！值为: 0.1521
[Component-wise GBSA] 评估完毕 -> 混合分数: 0.8604


### 模型预测：Fast Survival SVM

In [26]:
from sksurv.svm import FastSurvivalSVM
from sksurv.linear_model import CoxPHSurvivalAnalysis

print("Training Fast Survival SVM...")

class CalibratedFastSurvivalSVM:
    def __init__(self, **kwargs):
        self.model = FastSurvivalSVM(**kwargs)
        # 用外挂的 Cox 基准模型，让只有排序得分的 SVM 也能还原出生存概率曲线
        self._calibrator = CoxPHSurvivalAnalysis()

    def fit(self, X, y):
        self.model.fit(X, y)
        preds = self.model.predict(X).reshape(-1, 1)
        self._calibrator.fit(preds, y)
        return self

    def predict(self, X):
        return self.model.predict(X)

    def predict_survival_function(self, X):
        preds = self.model.predict(X).reshape(-1, 1)
        return self._calibrator.predict_survival_function(preds)

# SVM 对特征尺度非常敏感，使用 StandardScaler 的数据
svm_calibrated = CalibratedFastSurvivalSVM(max_iter=1000, tol=1e-5, random_state=42)
svm_calibrated.fit(X_train_scaled, y_train)
evaluate_model('Fast Survival SVM', svm_calibrated, X_train_scaled, y_train, y_train)

Training Fast Survival SVM...
[Fast Survival SVM] Brier 分数计算成功！值为: 0.3120
[Fast Survival SVM] 评估完毕 -> 混合分数: 0.7529


### 模型预测：Single Survival Tree

In [27]:
from sksurv.tree import SurvivalTree
print("Training Single Survival Tree...")
# 单一决策树基线模型
stree = SurvivalTree(max_depth=10, min_samples_split=20, min_samples_leaf=10, random_state=42)
stree.fit(X_train, y_train)
evaluate_model('Single Survival Tree', stree, X_train, y_train, y_train)

Training Single Survival Tree...
[Single Survival Tree] Brier 分数计算成功！值为: 0.0716
[Single Survival Tree] 评估完毕 -> 混合分数: 0.9350


### 模型预测：XGBoost (survival:cox)

In [ ]:
import xgboost as xgb
from sksurv.linear_model import CoxPHSurvivalAnalysis

print("Training XGBoost (survival:cox)...")
# XGBoost 原生支持 survival:cox，标签格式：发生事件为正时间，截尾为负时间
y_xgb = np.where(train_df['event'], train_df['time_to_hit_hours'], -train_df['time_to_hit_hours'])

# 自定义 Wrapper，使用 Cox 模型来转换风分(Risk Score)以计算出真实的存活概率曲线 (Survival Function)
class NativeXGBoostSurvival:
    def __init__(self, **kwargs):
        self.model = xgb.XGBRegressor(objective='survival:cox', tree_method='hist', **kwargs)
        self._calibrator = CoxPHSurvivalAnalysis() # 用于将风险得分转为生存概率曲线
    
    def fit(self, X, y_xgb_format, y_sksurv_format):
        self.model.fit(X, y_xgb_format)
        # 将模型输出当成单一特征，外挂一个简单的 Cox 回归得到基础生存分布
        preds = self.model.predict(X).reshape(-1, 1)
        self._calibrator.fit(preds, y_sksurv_format)
        return self
    
    def predict(self, X):
        return self.model.predict(X)
        
    def predict_survival_function(self, X):
        preds = self.model.predict(X).reshape(-1, 1)
        return self._calibrator.predict_survival_function(preds)
    
xgb_cox = NativeXGBoostSurvival(n_estimators=50, learning_rate=0.1, random_state=42)
# 这里特别传入了 `y_train` 给到 calibrator 
xgb_cox.fit(X_train, y_xgb, y_train)
evaluate_model('XGBoost (survival:cox)', xgb_cox, X_train, y_train, y_train)

Training XGBoost (survival:cox)...
[XGBoost (survival:cox)] Brier 分数计算成功！值为: 0.1543
[XGBoost (survival:cox)] 评估完毕 -> 混合分数: 0.8902


## 3. 对比总结

In [29]:
results_df = pd.DataFrame(results_table)
display(results_df.sort_values(by='混合分数', ascending=False))

,模型名称,C 指数,加权布里尔评分,混合分数
1,Gradient Boosting Survival / XGBoost,0.968864,0.046217,0.958307
7,Single Survival Tree,0.950439,0.071555,0.935043
0,Random Survival Forest,0.927335,0.080540,0.921822
2,DeepSurv / MLP (Proxy),0.903528,0.088243,0.909288
8,XGBoost (survival:cox),0.993914,0.154253,0.890197
5,Component-wise GBSA,0.889450,0.152088,0.860374
3,Extra Survival Trees,0.865477,0.158099,0.848974
4,Cox Proportional Hazards,0.904439,0.312032,0.752909
6,Fast Survival SVM,0.904439,0.312032,0.752909


### 模型评分指标解读：

1. **C 指数 (Concordance Index)**
👉 **越高越好，范围理论上在 0~1 之间。**
- 衡量模型排序风险的能力。任取二次火灾事故，只要模型在存活得更长的那次给出的风险分数更低，就算答对。
- **0.5 以下或左右**：等同于瞎猜；**0.6 ~ 0.7**：及格；**0.7 ~ 0.8**：优秀。

2. **加权布里尔评分 (Weighted Brier Score)**
👉 **越低越好，范围在 0~1 之间。**
- 衡量“模型预测在特定时刻（24/48/72H）的生存概率”与真实情况的均方误差。
- **0.0**：完美；**0.05 ~ 0.10**：顶级概率标定预测；**0.25**：永远猜 50% 得到的基线分水岭；**>0.25**：说明不仅猜错且极其“自信地猜错”（如 Cox 模型）。

3. **混合分数 (Mixed Score)**
👉 **越高越好。由自定义权重结合算得（0.3 × C指 + 0.7 × (1 - Brier)）。**
- 用于综合评估模型的**相对排序能力（C-Index）**与**绝对概率准确性（Brier Score）**。

**🔥 总结：C指数和混合分数要“往高了走（越近 1 越好）”，布里尔评分要“往低了压（越近 0 越好）”。在本次火灾数据表现下，树模型（尤其是 Gradient Boosting Survival / XGBoost）得益于其出色的局部概率标定能力，展现出了压倒性的统治得分。**